# Strong Allee effect and harvesting

Assembled from five task fragments. Read the scope/status in each section. Synthetic examples are not empirical observations. The final submission requires completed derivations, algorithms, results, interpretation and AI/data acknowledgements.

In [ ]:
import os
from pathlib import Path

import mm_allee

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src/mm_allee").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout")
os.chdir(ROOT)
print("Package:", mm_allee.__version__)

## 1. Foundations and Model A

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP01.</strong> Units and assumptions; logistic baseline; nondimensionalisation; A equilibria and stability, phase lines and branch plot.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP01: task not implemented yet; see docs/tasks/")

## 2. Model B, comparison and harvesting history

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP02.</strong> B roots/stability; fair A/B comparisons; up/down ramp with state carry-over; explicit restocking; no invented automatic recovery.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP02: task not implemented yet; see docs/tasks/")

## 3. Collapse thresholds, extinction and basins

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP03.</strong> Threshold derivations; finite versus asymptotic extinction; quadrature/event validation; basin maps; synthetic reference provenance.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP03: task not implemented yet; see docs/tasks/")

## 4. Forward PINNs

**Question.** If a small neural network is given only the ODE and the initial condition, how well does it predict the population, compared with a standard solver? All data are *synthetic and dimensionless* ($x=N/K$, $\tau=rt$).

**Equations.** Model A: $\dot x=G(x)-q$; Model B: $\dot x=G(x)-ex$; $G(x)=x(1-x)(x/a-1)$. Write $\dot x=f(x)$.

**Network with a hard initial condition.** A fully connected tanh network $N(s;\theta)$ with layer sizes $(1,32,32,1)$ (JAX, float64) and scaled time $s=\tau/T$ ($T$ fixed) defines

$$\hat x(\tau)=x_0+s\,N(s;\theta),$$

so $\hat x(0)=x_0$ **exactly for any weights**: no initial-condition loss is needed. Automatic differentiation with respect to physical time applies the chain rule $d\hat x/d\tau=T^{-1}\,d\hat x/ds$ (with the product rule on $s\,N$).

**Loss.** At $n=100$ fixed uniform collocation times $\tau_i\in[0,T]$:

$$R(\tau)=\frac{d\hat x}{d\tau}-f(\hat x),\qquad \mathcal L(\theta)=\frac1n\sum_i R(\tau_i)^2 .$$

**Algorithm.** (1) fixed collocation grid; (2) $\hat x$ and $d\hat x/d\tau$ by `jax.grad` + `vmap`; (3) $\mathcal L$; (4) Adam for 1000 iterations (about $2/7$ of the 3500-iteration budget) to reach a good region, then L-BFGS for 2500 iterations to converge. No solver solution is used as a label. For A the training interval ends at the extinction time, so the zero tail never enters the residual.

**Evaluation.** Dense DOP853 reference (rtol $10^{-12}$), separate from training. Metrics: RMSE, MAE, global relative $L^2$ (reported in scientific notation), max error, negative samples, runtime. A run is *inaccurate* if relative $L^2>5\times10^{-2}$ (threshold fixed beforehand).


In [ ]:
# Live demo (a few seconds on CPU): short run (100 Adam + 500 L-BFGS), Model B regular, seed 11.
# In the recorded run this reaches a tiny physics loss (about 4e-6) but a LARGE error (relative L2
# about 9e-1): a spurious zero-residual solution. Compare the printed loss with the printed error.
import sys

sys.path.insert(0, "experiments/wp04")
import run as wp04

from mm_allee.pinn.train_forward import evaluate_prediction, predict

case = dict(model="B", a=0.2, harvest=0.3, x0=0.8, tau_end=10.0)
ref = dict(method="DOP853", rtol=1e-12, atol=1e-14, n_points=1001)
tau, x_ref, hit = wp04.reference_trajectory(case, ref)
res = wp04.run_pinn(case, 11, {"training": {"adam_iters": 100, "lbfgs_iters": 500}}, hit)
ev = evaluate_prediction(predict(res.params, res.config, tau), x_ref)
print({k: (f"{v:.3e}" if isinstance(v, float) else v) for k, v in ev.items()})
print(f"{res.wall_seconds:.1f} s, final physics loss {res.history['loss'][-1]:.3e}")

## Curated final results (loaded, not retrained)

The table is `artifacts/tables/wp04_forward_metrics.csv`, produced by
`python experiments/wp04/run.py --config configs/wp04_final.json --output results/wp04 --artifacts artifacts` (about 1 min on CPU; see `artifacts/manifests/wp04_forward.json`).


In [ ]:
import csv

rows = list(csv.DictReader(open("artifacts/tables/wp04_forward_metrics.csv")))


def fmt(v):
    return f"{float(v):.2e}" if v else "-"


print(
    f"{'case':13s}{'method':7s}{'seed':5s}{'status':12s}{'rel L2':>10s}{'max err':>10s}{'neg':>5s}{'sec':>7s}"
)
for r in rows:
    print(
        f"{r['case']:13s}{r['method']:7s}{r['seed']:5s}{r['status']:12s}{fmt(r['relative_l2']):>10s}"
        f"{fmt(r['max_abs']):>10s}{r['boundary_violations']:>5s}{float(r['wall_seconds']):7.2f}"
    )

![Forward PINN vs reference](../artifacts/figures/wp04_forward.png)

![Error and loss](../artifacts/figures/wp04_error_loss.png)

Curated PNGs from the final run (EPS copies are in `artifacts/figures/`). The vertical line in the bottom row marks the Adam to L-BFGS hand-over.

## Interpretation

- **Accurate for every seed:** A subthreshold extinction ($10^{-7}$ relative $L^2$), B near-fold ($10^{-7}$) and B subthreshold decay ($10^{-5}$ to $10^{-4}$).
- **B regular fails for all three seeds** ($6.8\times10^{-1}$ to $9.2\times10^{-1}$) even though the physics loss is only $2\times10^{-8}$ to $3\times10^{-6}$. Seeds 11 and 22 end at $x\approx0.2838$, the unstable equilibrium $x_-=0.28377$, and seed 33 decays toward the equilibrium $0$: at an equilibrium $f=0$, so the residual is zero. These are *spurious zero-residual solutions*. The hard initial condition fixes $\hat x(0)$ but does not prevent them. **A small loss is therefore not evidence of an accurate trajectory.**
- **A regular:** seed 22 is accurate ($1.9\times10^{-5}$); seeds 11 and 33 fail ($7.6\times10^{-1}$, $8.5\times10^{-1}$; seed 11 sits near $x_-=0.29598$, seed 33 has not converged). **A near-fold:** seed 11 is inaccurate ($9.1\times10^{-2}$), the others are accurate.
- 12 of 18 runs are accurate, 6 are not, none diverged. The L-BFGS phase does most of the work: the loss plateaus during Adam and drops by orders of magnitude only after the hand-over.
- RK45 matches the reference to about $10^{-10}$ in $3\times10^{-3}$ to $9\times10^{-3}$ s; each PINN run took $2.4$ to $3.6$ s. For a scalar ODE the classical solver is the better tool; the PINN's value is as the engine for inverse problems (Task 5).
- *Limitations:* one parameter set per regime, one untuned configuration, three seeds (failures are strongly seed dependent), one machine. With 100 collocation points a fast initial transient can fit between two points; this is consistent with the jumps but was not separately demonstrated. Denser early collocation or time-marching are untested.


## 5. Inverse PINNs and validation

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP05.</strong> Identifiable staged unknowns; noisy observations; ODE least-squares baseline; held-out prediction; near-collapse failures; AI/data acknowledgement.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP05: task not implemented yet; see docs/tasks/")